In [ ]:
import warnings
warnings.filterwarnings('ignore')
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import pip
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder
from sklearn.tree import DecisionTreeClassifier
from sklearn.metrics import confusion_matrix, accuracy_score, classification_report, roc_auc_score, precision_score, recall_score, f1_score, ConfusionMatrixDisplay
from sklearn.tree import export_graphviz


 

In [ ]:
dataset = pd.read_csv(r'C:\Users\User\Downloads\Bank Customer Churn Prediction.csv')

Analise Exploratória

In [ ]:
dataset.head()

In [ ]:
dataset.shape

In [ ]:
dataset.info()

In [ ]:
# Distribuição do churn
contagem = dataset['churn'].value_counts()
porcentagem = dataset['churn'].value_counts(normalize=True) * 100

print('Clientes que ficaram :', contagem[0], f'— {porcentagem[0]:.1f}%')
print('Clientes que saíram :', contagem[1], f'— {porcentagem[1]:.1f}%')

In [ ]:
fig, axes = plt.subplots(2, 3, figsize=(16, 9))
fig.suptitle('Relação das variáveis com Churn', fontsize=14, fontweight='bold')

# Idade
dataset.boxplot(column='age', by='churn', ax=axes[0, 0])
axes[0, 0].set_title('Idade')
axes[0, 0].set_xlabel('Churn (0=Ficou, 1=Saiu)')

# Score de crédito
dataset.boxplot(column='credit_score', by='churn', ax=axes[0, 1])
axes[0, 1].set_title('Score de Crédito')
axes[0, 1].set_xlabel('Churn')

# Saldo
dataset.boxplot(column='balance', by='churn', ax=axes[0, 2])
axes[0, 2].set_title('Saldo')
axes[0, 2].set_xlabel('Churn')

# Churn por país
churn_pais = dataset.groupby('country')['churn'].mean() * 100
churn_pais.plot(kind='bar', ax=axes[1, 0], color=['steelblue', 'tomato', 'steelblue'], edgecolor='black')
axes[1, 0].set_title('Taxa de Churn por País (%)')
axes[1, 0].tick_params(axis='x', rotation=0)

# Churn por número de produtos
churn_prod = dataset.groupby('products_number')['churn'].mean() * 100
churn_prod.plot(kind='bar', ax=axes[1, 1], color='steelblue', edgecolor='black')
axes[1, 1].set_title('Taxa de Churn por Nº de Produtos (%)')
axes[1, 1].set_xlabel('Produtos')
axes[1, 1].tick_params(axis='x', rotation=0)

# Churn por gênero
churn_gen = dataset.groupby('gender')['churn'].mean() * 100
churn_gen.plot(kind='bar', ax=axes[1, 2], color=['salmon', 'steelblue'], edgecolor='black')
axes[1, 2].set_title('Taxa de Churn por Gênero (%)')
axes[1, 2].tick_params(axis='x', rotation=0)

plt.tight_layout()
plt.savefig('eda_churn.png', dpi=150, bbox_inches='tight')
plt.show()

Conventer texto pra número

In [ ]:

df = dataset.drop(columns=['customer_id'])

# Converter texto para número
encoder = LabelEncoder()
dataset['country'] = encoder.fit_transform(dataset['country'])  # France=0, Germany=1, Spain=2
dataset['gender']  = encoder.fit_transform(dataset['gender'])   # Female=0, Male=1

# Confirmar
print('country =', dataset['country'].unique())
print('gender =', dataset['gender'].unique())

Dividindo variaveis de teste

In [ ]:
X = dataset.drop(columns=['churn'])  
y = dataset['churn']                 

print('Features (X):', list(X.columns))
print('Shape de X:', X.shape)
print('Shape de y:', y.shape)

Features (X): ['customer_id', 'credit_score', 'country', 'gender', 'age', 'tenure', 'balance', 'products_number', 'credit_card', 'active_member', 'estimated_salary']
Shape de X: (10000, 11)
Shape de y: (10000,)


In [56]:
X_train, X_test, y_train, y_test = train_test_split(
    X, y,
    test_size=0.3,    # 30% para teste, 70% para treino
    random_state=0,   # garante sempre o mesmo sorteio
    stratify=y        # mantém proporção de churn nos dois grupos
)

print(f'Treino : {X_train.shape[0]} clientes')
print(f'Teste  : {X_test.shape[0]} clientes')
print()
print(f'% de churn no treino : {y_train.mean():.3f}')
print(f'% de churn no teste  : {y_test.mean():.3f}')

Treino : 7000 clientes
Teste  : 3000 clientes

% de churn no treino : 0.204
% de churn no teste  : 0.204


Criação do modelo

In [57]:
arvore = DecisionTreeClassifier(
    max_depth=5,
    random_state=0,
    class_weight='balanced'
)

arvore.fit(X_train, y_train)

print('Modelo treinado.')
print(f'Profundidade final : {arvore.get_depth()}')
print(f'Número de folhas   : {arvore.get_n_leaves()}')

Modelo treinado.
Profundidade final : 5
Número de folhas   : 31
